## Imports

In [0]:
%pip install openpyxl

In [0]:
from datetime import date, datetime
from pyspark.sql.functions import lit, format_number, coalesce, split, current_timestamp, col
from pyspark.sql import SparkSession
from pyspark.dbutils import DBUtils
from io import BytesIO
import pandas as pd
import boto3
import json
import os

In [0]:
spark = SparkSession.builder.getOrCreate()
dbutils = DBUtils(spark)

# Function to get AWS credentials from Databricks secrets
def get_aws_credentials(secrets_scope, key_name):
    try:
        # Databricks secrets configuration
   
        aws_secret_key = dbutils.secrets.get(scope=secrets_scope, key=key_name)
        if aws_secret_key is not None:
            print("AWS secret key retrieved successfully.")
        else:
            print("AWS secret key is None. Please check Databricks secrets configuration.")
 
        return aws_secret_key
    except Exception as e:
        print(f"Error retrieving AWS credentials: {str(e)}")
        return None

# Function to get AWS credentials from Databricks secrets
def get_scope_key_names(secrets_scope):
    try:
        # Databricks secrets configuration
 
        aws_keys = dbutils.secrets.list(scope=secrets_scope)
        key_names = [key.key for key in aws_keys] if aws_keys else []
        return key_names
   
    except Exception as e:
        print(f"Error retrieving AWS keys: {str(e)}")
        return None    

# Function to connect to AWS and access SSM parameter
def connect_to_aws_and_access_ssm(ssm_key, ssm_client, aws_secret_key):
 
    if aws_secret_key is None:
        print("AWS secret key not available.")
        return
 
    # Access SSM parameter /cfn/a360/reports_output_path
    response = ssm_client.get_parameter(
        Name=ssm_key,
        WithDecryption=False
    )
 
    return response['Parameter']['Value']

# Function to get the environment prefix
def get_env_prefix():
    secrets_scope = "aws-databricks"
    aws_keys = get_scope_key_names(secrets_scope)
    aws_key = aws_keys[0] if aws_keys else None
    # Get AWS secret key from Databricks secrets
    aws_secret_key = get_aws_credentials(secrets_scope,aws_key)
    # Connect to AWS using boto3
    ssm_client = boto3.client('ssm', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)
 
    S3_client = boto3.client('s3', region_name='eu-central-1',
                            aws_access_key_id=aws_key,
                            aws_secret_access_key=aws_secret_key)  

    env_prefix = connect_to_aws_and_access_ssm('/cfn/bucket_prefix', ssm_client, aws_secret_key)
    return env_prefix

## Get the needed Perimeter and Parameters

In [0]:
env = get_env_prefix()

In [0]:
try:    
    report_args = dbutils.widgets.get("report_args")
    report_args = report_args.replace("\'", "\"")
except Exception as e:
    raise ValueError("Missing required input: report_args")

# Read JSON
json_data = json.loads(report_args)
 
# Access the values and store them into variables
if "output_file_prefix" not in json_data: raise KeyError("'output_file_prefix' key is missing in the report_args JSON data.")
if "report_output" not in json_data: raise KeyError("'report_output' key is missing in the report_args JSON data.")

output_file_prefix = "".join(json_data['output_file_prefix'])
report_output = json_data['report_output']

In [0]:
# --- Execution mode: "portfolio" (original) or "perimeter" (offboarding/file-based) ---
try:
    execution_mode = dbutils.widgets.get("execution_mode")
except:
    execution_mode = "portfolio"
if not execution_mode:
    execution_mode = "portfolio"
if execution_mode not in ("portfolio", "perimeter"):
    raise ValueError("execution_mode must be 'portfolio' or 'perimeter'")

portfolioId = dbutils.widgets.get("portfolioId")
if execution_mode == "portfolio" and not portfolioId:
    raise ValueError("Missing required input: portfolioId (required in portfolio mode)")

run_date = dbutils.widgets.get("run_date")

run_date = spark.sql(f"""                    
                        select 
                            case 
                                when try_cast('{run_date}' as date) is not null then try_cast('{run_date}' as date)
                                else PreviousBusinessDay
                            end as BusinessDate
                        from `{env}-cdh`.reference.ref_calendar
                        where CalendarDate = coalesce(try_cast('{run_date}' as date),cast(getdate() as date))
                    """).first()[0]

print(f"Execution mode: {execution_mode} | Run date: {run_date}")

## Read Source Tables

In [0]:
dparty = spark.sql(f"""
                            select distinct
                                      dp.CDI
                                    , FirstNameGR
                                    , LastNameGR
                                    , PartyType
                                    , LegalNameGR
                                    , TaxRegistrationNumber
                                    , DeceasedFlag
                                    , NationalNumber
                                    , FirstFatherNameGR
                                    , PartykeyProfits
                                    , LegalForm
                            from `{env}-cdh`.derivative.dparty dp
                            where BusinessDate = '{run_date}'
                        """)

dparty.createOrReplaceTempView("dparty")

In [0]:
dexposure = spark.sql(f"""
                            select distinct
                                      Exposurekey
                                    , Agreementkey
                                    , OutstandingTotalBalance
                                    , OnBalance
                                    , OffBalance
                                    , AssetSegment
                                    , PortfolioName
                                    , TotalPrincipal
                                    , TotalInterestAccruals
                                    , TotalExpense
                                    , OutstandingTotalBalance
                                    , Currency
                                    , ExposureStatus
                                    , IsActiveFlag
                                    , TotalInterestL3869
                                    , IsActiveFlag
                                    , BusinessDate
                            from `{env}-cdh`.derivative.dexposure dex
                            where BusinessDate = '{run_date}'
                        """)

dexposure.createOrReplaceTempView("dexposure")

In [0]:
InitialExposure = spark.sql(f"""
                                select distinct
                                          Exposurekey
                                        , CDI
                                        , PartyRelationshipType
                                        , CaseCode
                                        , DiasNumber
                                        , SourceSystem
                                        , BusinessDate
                                from `{env}-cdh`.primary.exposure ex
                            """)

InitialExposure.createOrReplaceTempView("InitialExposure")

In [0]:
CaseCode = spark.sql(f"""                    
                        SELECT ExposureKey, CDI, CaseCode
                        FROM (
                            SELECT ExposureKey, CaseCode, CDI,
                                    ROW_NUMBER() OVER (PARTITION BY ExposureKey, CDI ORDER BY BusinessDate DESC) AS rn
                            FROM InitialExposure
                            WHERE SourceSystem in ('qc', 'qca')
                            and CaseCode is not null
                        ) t
                        WHERE rn = 1
                    """)

CaseCode.createOrReplaceTempView("CaseCode")

In [0]:
exposure = spark.sql(f"""
                            select distinct
                                      ex.Exposurekey
                                    , dex.Agreementkey
                                    , ex.CDI
                                    , dex.PortfolioName
                                    , ex.PartyRelationshipType
                                    , cc.CaseCode
                                    , ex.DiasNumber
                                    , ex.SourceSystem
                            from InitialExposure ex
                            inner join dexposure dex
                            on dex.ExposureKey = ex.ExposureKey
                            and ex.BusinessDate = dex.BusinessDate
                            left join CaseCode cc
                            on cc.CDI = ex.CDI
                            and (ex.ExposureKey = cc.ExposureKey or substring(ex.ExposureKey, 4, 50) = cc.ExposureKey)
                        """)

exposure.createOrReplaceTempView("exposure")

In [0]:
dagreement = spark.sql(f"""
                        select distinct
                                  Agreementkey
                                , AgreementStatus
                        from `{env}-cdh`.derivative.dagreement
                        where BusinessDate = '{run_date}'
                    """)

dagreement.createOrReplaceTempView("dagreement")

In [0]:
masterXslave = spark.sql(f"""
                            select distinct
                                      EntityKey
                                    , RelatedEntityKey
                            from `{env}-cdh`.derivativeactive.dentityrelationships
                            where BusinessDate = '{run_date}'
                            and EntityRelationshipType = 'Master - Slave Rltnp'
                        """)

masterXslave.createOrReplaceTempView("masterXslave")

In [0]:
master = spark.sql(f"""
                        select distinct EntityKey MASTER_ACC
                        from masterXslave
                    """)

master.createOrReplaceTempView("master")

In [0]:
slave = spark.sql(f"""
                        select distinct RelatedEntityKey SLAVE_ACC
                        from masterXslave
                    """)

slave.createOrReplaceTempView("slave")

In [0]:
# In portfolio mode: filter by portfolioId
# In perimeter mode: load all portfolios (filtering is done via offboarding_perimeter join)
if execution_mode == "portfolio":
    portfolio_filter = f"and PortfolioId in ({','.join(portfolioId.split(','))})"
else:
    portfolio_filter = ""

Portfolio = spark.sql(f"""
                            select distinct
                                      PortfolioId
                                    , PortfolioName
                                    , Prefix
                            from `{env}-cdh`.primary.Portfolio
                            where BusinessDate = '{run_date}'
                            {portfolio_filter}
                        """)

Portfolio.createOrReplaceTempView("Portfolio")

In [0]:
ContactDetails = spark.sql(f"""
                            select distinct
                                      CDI 
                                    , Street
                                    , PostalCode
                                    , City
                                    , Country
                                    , CommunicationFlag
                                    , ContactDetailType
                            from `{env}-cdh`.primary.ContactDetails
                            where BusinessDate = '{run_date}'
                            and Sourcesystem = 'profits'
                        """)

ContactDetails.createOrReplaceTempView("ContactDetails")

In [0]:
# Load perimeter only in perimeter mode
# Supports: (a) reference table, (b) Excel/CSV file from volume
if execution_mode == "perimeter":
    try:
        perimeter_file = dbutils.widgets.get("perimeter_file") or ""
    except:
        perimeter_file = ""
    
    if perimeter_file:
        if not perimeter_file.endswith('.xlsx'):
            raise ValueError(f"Το perimeter αρχείο πρέπει να είναι .xlsx. Δόθηκε: '{perimeter_file}'")

        perimeter_path = f"/Volumes/{env}-cdh-delivery/internalreports/goodbyeletter/input/{perimeter_file}"
        pandas_df = pd.read_excel(perimeter_path)
        perimeter_df = spark.createDataFrame(pandas_df.astype(str))

        cols = perimeter_df.columns
        cols_lower = [col.lower() for col in cols]
        cdi_col = next((c for c in ['cdi', 'cif'] if c in cols_lower), None)
        exposurekey_col = next((c for c in ['exposurekey'] if c in cols_lower), None)
        agreementkey_col = next((c for c in ['agreementkey'] if c in cols_lower), None)
        
        if cdi_col and (exposurekey_col or agreementkey_col):
            if "exposurekey" in cols_lower:
                offboarding_perimeter = perimeter_df.select(
                    col(cdi_col).cast("String").alias("CDI"),
                    col(exposurekey_col).cast("String").alias("ExposureKey")
                    ).distinct()
            
            elif "agreementkey" in cols_lower:
                perimeter_dt = perimeter_df.select(
                    col(cdi_col).cast("String").alias("CDI"),
                    col(agreementkey_col).cast("String").alias("AgreementKey")
                    ).distinct()
                
                perimeter_dt.createOrReplaceTempView("perimeter_dt")

                # 1. Based on this query, the Exclusion file contains only Open Contract Numbers. 
                #    Therefore, applying the filter `Exclusion Reason = 'Closed Loan'` will return no rows.
                offboarding_perimeter = spark.sql(f"""
                                                  
                    WITH exposure AS (
                        SELECT SourceSystem, 
                               PortfolioName,
                               AgreementKey,
                               ExposureKey,
                               CDI,
                               PartyRelationshipType, 
                               IsActiveFlag,
                               BusinessDate 
                        FROM `{env}-cdh`.primary.exposure
                        WHERE BusinessDate = '{run_date}'
                        AND LOWER(SourceSystem) = 'profits'
                    ),

                    offboardingPerimeterWithExposureKey AS (
                        SELECT y.SourceSystem, 
                                y.PortfolioName, 
                                x.AgreementKey, 
                                y.ExposureKey, 
                                y.CDI,  
                                y.PartyRelationshipType, 
                                y.IsActiveFlag, 
                                y.BusinessDate
                        FROM  perimeter_dt AS x
                        INNER JOIN exposure AS y
                        ON x.AgreementKey = y.AgreementKey
                        )

                    SELECT DISTINCT ExposureKey, CDI
                    FROM offboardingPerimeterWithExposureKey                                       
                    """)
                
                # 2. Based on this query, the Exclusion file contains Open as well as Closed Contract Numbers.
                #    Therefore, applying the filter `Exclusion Reason = 'Closed Loan'` will return many rows.
                offboarding_perimeter = spark.sql(f"""
                                                  
                    WITH withProfitsExposure AS (
                        SELECT SourceSystem,
                                PortfolioName,
                                AgreementKey,
                                ExposureKey,
                                CDI, 
                                PartyRelationshipType,
                                IsActiveFlag,
                                BusinessDate
                        FROM `{env}-cdh`.primary.exposure
                        WHERE LOWER(SourceSystem) = 'profits'
                    ),            
                    
                        withMaxBusinessDate AS (
                            SELECT SourceSystem,
                                   PortfolioName,
                                   AgreementKey,
                                   ExposureKey,
                                   CDI, 
                                   MAX(BusinessDate) AS MaxBusinessDate
                            FROM withProfitsExposure
                            GROUP BY SourceSystem,
                                     PortfolioName,
                                     AgreementKey,
                                     ExposureKey,
                                     CDI
                            ),

                        exposureWithMaxBusinessDate AS (
                            SELECT x.*
                            FROM withProfitsExposure AS x
                            INNER JOIN withMaxBusinessDate AS y
                            ON x.BusinessDate = y.MaxBusinessDate
                            AND x.SourceSystem = y.SourceSystem
                            AND x.PortfolioName = y.PortfolioName
                            AND x.AgreementKey = y.AgreementKey
                            AND x.ExposureKey = y.ExposureKey
                            AND x.CDI = y.CDI
                            ),

                        offboardingPerimeterWithExposureKey AS (
                            SELECT y.SourceSystem, 
                                y.PortfolioName, 
                                x.AgreementKey, 
                                y.ExposureKey, 
                                y.CDI,  
                                y.PartyRelationshipType, 
                                y.IsActiveFlag, 
                                y.BusinessDate
                            FROM  perimeter_dt AS x
                            LEFT JOIN exposureWithMaxBusinessDate AS y
                            ON x.AgreementKey = y.AgreementKey
                            )

                    SELECT DISTINCT ExposureKey, CDI
                    FROM offboardingPerimeterWithExposureKey
                    """)
 
            else:
                raise ValueError("Neither ExposureKey nor AgreementKey exists in perimeter file")
        else:
            raise ValueError(f"Perimeter file must have CDI and ExposureKey or AgreementNumber columns. Found: {cols}")
        print(f"Perimeter loaded from file: '{perimeter_file}'")

    else:
        # Reference table perimeter (default: Jupiter offboarding)
        try:
            perimeter_table = dbutils.widgets.get("perimeter_table") or "`d-cdh`.reference.jupiter_offboarding_cdi_exposure_20260622"
        except:
            perimeter_table = "`d-cdh`.reference.jupiter_offboarding_cdi_exposure_20260622"
        offboarding_perimeter = spark.sql(f"""SELECT DISTINCT cdi, exposurekey FROM {perimeter_table}""")
        print(f"Perimeter loaded from table: {perimeter_table}")
    
    offboarding_perimeter.createOrReplaceTempView("offboarding_perimeter")
else:
    print("Portfolio mode — no perimeter loaded")

## Create base tables

In [0]:
LoanFinancialsContract = spark.sql(f"""
                                        select distinct 
                                                  ContractID
                                                , ContractNumber 
                                                , SUM(OutstandingTotalBalance) OutstandingTotalBalance
                                                , SUM(OnBalance) OnBalance
                                                , SUM(OffBalance) OffBalance
                                        from (
                                            select distinct 
                                                    Agreementkey ContractID
                                                    , substring(Agreementkey,4,99) ContractNumber
                                                    , OutstandingTotalBalance
                                                    , OnBalance
                                                    , OffBalance
                                            from dexposure
                                        ) base
                                        group by ContractID, ContractNumber
                                    """)

LoanFinancialsContract.createOrReplaceTempView("LoanFinancialsContract")

In [0]:
LoanFinancialsAccount = spark.sql(f"""
                                        select distinct 
                                                  Exposurekey
                                                , AccountNumber 
                                                , SUM(OutstandingTotalBalance) OutstandingTotalBalance
                                                , SUM(OnBalance) OnBalance
                                                , SUM(OffBalance) OffBalance
                                        from (
                                            select distinct 
                                                      Exposurekey
                                                    , substring(Exposurekey,4,99) AccountNumber
                                                    , OutstandingTotalBalance
                                                    , OnBalance
                                                    , OffBalance
                                            from dexposure
                                        ) base
                                        group by Exposurekey, AccountNumber
                                    """)

LoanFinancialsAccount.createOrReplaceTempView("LoanFinancialsAccount")

In [0]:
base = spark.sql(f"""
                    select distinct
                              a.*
                            , lfc.OutstandingTotalBalance `Total Legal Claim Contract`
                            , lfc.OnBalance
                            , lfc.OffBalance
                            , lfa.OutstandingTotalBalance `Total Legal Claim Account`
                    from (
                        select distinct
                                row_number() over (partition by dp.CDI, dex.Exposurekey 
                                                       order by case 
                                                                    when ex.PartyRelationshipType like 'Primary%' then 1 
                                                                    when ex.PartyRelationshipType like 'Guarantor%' then 2 
                                                                    else 3 
                                                                end) rankrelation
                                , p.PortfolioId
                                , p.PortfolioName
                                , dp.CDI CIF
                                , dp.taxregistrationnumber AFM
                                , dp.NationalNumber `ΑΔΤ`
                                , dp.LegalNameGR `Εκτυπώσιμη Επωνυμία`
                                , case 
                                    when dp.PartyType = 'Individual' then dp.FirstFatherNameGR 
                                    else NULL 
                                end as `Father Name`
                                , dp.DeceasedFlag
                                , ex.CaseCode
                                , substring(dex.Agreementkey, 4, 50) ContractNumber
                                , substring(dex.Exposurekey, 4, 50) AccountNumber
                                , case dex.AssetSegment
                                    when 'CL' then 'Καταναλωτικό' 
                                    when 'ML' then 'Στεγαστικό' 
                                    else 'Επιχειρηματικό'
                                  end ProductType
                                , dp.FirstNameGR `First Name`
                                , dp.LastNameGR `Last Name`
                                , concat('Loan-', ex.PartyRelationshipType, ' Rltp') RelationshipType
                                , CASE 
                                    WHEN LegalForm = 'ΙΔΙΩΤΕΣ - ΑΤΟΜΙΚΕΣ ΕΠΙΧΕΙΡΗΣΕΙΣ' AND dp.partytype = 'Organization' THEN 'Yes'
                                    ELSE 'No'
                                  END `Ένδειξη Ατομικής επιχείρησης`
                                , dex.ExposureStatus AccountStatus
                                , case 
                                    when s.SLAVE_ACC is not NULL then 'Slave'
                                    when m.MASTER_ACC is not NULL then 'Master'
                                    else 'Other'
                                  end as `Account Type (Master / Slave / Other)`
                                , case
                                    when dagr.Agreementkey is null or AgreementStatus = 'Closed' then 'Inactive'
                                    else 'Active'
                                  end `Agreement Status (Active/Inactive)`
                                , PartyType LegalEntity
                        from dparty dp
                        inner join exposure ex
                            on dp.CDI = ex.CDI
                        inner join dexposure dex
                            on dex.Exposurekey = ex.Exposurekey
                            and dex.IsActiveFlag = true
                        inner join Portfolio p
                            on p.PortfolioName = dex.PortfolioName
                        {"inner join offboarding_perimeter op on dp.CDI = op.cdi AND dex.Exposurekey = op.exposurekey" if execution_mode == "perimeter" else ""}
                        left join dagreement dagr
                            on dagr.Agreementkey = dex.Agreementkey
                        left join master m
                            on dex.Exposurekey = m.MASTER_ACC
                        left join slave s
                            on dex.Exposurekey = s.SLAVE_ACC
                    )  as a 
                    inner join LoanFinancialsContract lfc
                    on lfc.ContractNumber = a.ContractNumber
                    inner join LoanFinancialsAccount lfa
                    on lfa.AccountNumber = a.AccountNumber
                    where a.rankrelation = 1
                    order by CIF
                """)

base.createOrReplaceTempView("base")

In [0]:
baserank = spark.sql(f"""
                        select distinct
                                  CIF
                                , AFM
                                , `ΑΔΤ`
                                , `Εκτυπώσιμη Επωνυμία`
                                , `Father Name`
                                , DeceasedFlag
                                , ContractNumber
                                , AccountNumber
                                , productType
                                , LegalEntity
                                , `Total Legal Claim Contract`
                                , `Total Legal Claim Account`
                                , `Account Type (Master / Slave / Other)`
                                , OnBalance
                                , OffBalance
                                , PortfolioName
                                , CaseCode
                                , `First Name`
                                , `Last Name`
                                , RelationshipType
                                , `Ένδειξη Ατομικής επιχείρησης`
                                , AccountStatus
                                , `Agreement Status (Active/Inactive)`
                                , Row_number() over (partition by PortfolioName, CIF order by contractnumber, AccountNumber) AS rankd -- CHANGE 27/10
                        from base
                        {"where `Account Type (Master / Slave / Other)` != 'Slave' and `Agreement Status (Active/Inactive)` != 'Inactive'" if execution_mode == "perimeter" else ""}
                    """)

baserank.createOrReplaceTempView("baserank")
baserank_pd = baserank.toPandas()

In [0]:
CDI = spark.sql(f"""
                    select distinct
                              CIF
                            , PortfolioName
                            , AFM
                            , `ΑΔΤ`
                            , `Εκτυπώσιμη Επωνυμία`
                            , `First Name`
                            , `Last Name`
                            , `Father Name`
                            , DeceasedFlag
                            , LegalEntity
                            , `Ένδειξη Ατομικής επιχείρησης`
                    from baserank
                """)

CDI.createOrReplaceTempView("CDI")

In [0]:
max_rank = spark.sql(f"""
                    select MAX(rankd)
                    from baserank
                """).collect()[0][0]

In [0]:
# Define the fixed columns (they will NOT be pivoted)
Fixed_columns = [
    "PortfolioName", # Added 27/10
      "CIF"
]

# Define the pivotable columns (they will be pivoted)
Pivoted_columns = [
      "productType"
    , "ContractNumber"
    , "AccountNumber"
    , "Total Legal Claim Contract"
]

# Define the mapping for renaming columns
rename_mapping = {
    "productType": "Είδος χορήγησης",
    "ContractNumber": "APPLICATION NUMBER (contract-application)",
    "Total Legal Claim Contract": "Total Legal Claim Contract",
    "AccountNumber": "Κωδικός πληρωμής ανά Λογαριασμό (Account number)"
}


Pivot_index = ["CIF", "PortfolioName"] # Added 27/10

In [0]:
# Implementation 27/10
# Fixed columns
df_fixed = baserank_pd[Fixed_columns].drop_duplicates()

# Pivot each column in the Pivoted_columns list
df_pivoted = pd.DataFrame()

for col in Pivoted_columns:
    df_temp = baserank_pd.pivot(index=Pivot_index, columns='rankd', values=col)
    new_col_name = rename_mapping.get(col, col)
    df_temp.columns = [f'{new_col_name} {i}' for i in df_temp.columns]
    df_temp.reset_index(inplace=True)
    df_pivoted = pd.merge(df_pivoted, df_temp, on=Pivot_index, how='outer') if not df_pivoted.empty else df_temp

# Fixed + Pivoted columns
CDIContract_pd = pd.merge(df_pivoted, df_fixed, on=Pivot_index)

# Reorder the columns in the final DataFrame
pivoted_columns_order = [f'{rename_mapping.get(col, col)} {i}' for i in range(1, max_rank + 1) for col in Pivoted_columns]
final_column_order = Fixed_columns + pivoted_columns_order
CDIContract_pd = CDIContract_pd[final_column_order]

# Create the final DataFrames
CDIContract_pivoted_pd = CDIContract_pd[['CIF', 'PortfolioName'] + pivoted_columns_order].rename(columns={'CIF': 'CIF_pivoted', 'PortfolioName': 'PortfolioName_pivoted'})
CDIContract_pivoted = spark.createDataFrame(CDIContract_pivoted_pd)
CDIContract_pivoted.createOrReplaceTempView("CDIContract_pivoted")

In [0]:
Address = spark.sql(f"""
                        select *
                        from (
                            select distinct
                                      d.CIF
                                    , Street `Εκτυπώσιμη Διεύθυνση Επικοινωνίας`
                                    , PostalCode `ΤΚ`
                                    , City `Περιοχή - Πόλη`
                                    , Country `Χώρα`
                                    , rank() over (partition by d.CIF 
                                                order by CommunicationFlag desc
                                                        , case 
                                                            when ContactDetailType = 'Head Office' then 1 
                                                            when ContactDetailType = 'Work Address' then 2 
                                                            else 3 
                                                            end
                                                        ) rnk
                            from baserank d
                            inner join ContactDetails c
                            on d.CIF = c.CDI
                        ) as a
                        where rnk = 1
                    """)

Address.createOrReplaceTempView("Address")

## Horizontal File

In [0]:
Horizontal_file = spark.sql(f"""
                                select distinct
                                          c.CIF
                                        , c.AFM
                                        , c.`Εκτυπώσιμη Επωνυμία`
                                        , c.`First Name`
                                        , c.`Last Name`
                                        , c.`Father Name`
                                        , c.`LegalEntity` `Legal entity/ Individual`
                                        , c.`Ένδειξη Ατομικής επιχείρησης`
                                        , a.`Εκτυπώσιμη Διεύθυνση Επικοινωνίας`
                                        , a.`ΤΚ`
                                        , a.`Περιοχή - Πόλη`
                                        , a.`Χώρα`
                                        , c.DeceasedFlag
                                        , piv.*
                                from CDI c
                                inner join Address a
                                    on c.CIF = a.CIF
                                inner join CDIContract_pivoted piv
                                    on piv.CIF_pivoted = c.CIF
                                    and piv.PortfolioName_pivoted = c.PortfolioName
                            """)
                    
Horizontal_file = Horizontal_file.drop('CIF_pivoted')
Horizontal_file.createOrReplaceTempView("Horizontal_file")

## Vertical File

In [0]:
Vertical_file = spark.sql(f"""
                              select distinct
                                    b.PortfolioName
                                  , b.CIF
                                  , b.AccountNumber
                                  , b.ContractNumber
                                  , b.CaseCode
                                  , b.RelationshipType
                                  , b.`AccountStatus`
                                  , b.`Total Legal Claim Contract`
                                  , b.`Total Legal Claim Account`
                                  , b.LegalEntity
                                  , b.`Ένδειξη Ατομικής επιχείρησης`
                                  , b.`Account Type (Master / Slave / Other)`
                                  , b.DeceasedFlag
                              from baserank b
                          """)
                    
Vertical_file.createOrReplaceTempView("Vertical_file")

## Exclusion

In [0]:
base_exclusion = spark.sql(f"""
                        select distinct
                                  PortfolioName
                                , CIF
                                , AccountNumber
                                , ContractNumber
                                , CaseCode
                                , RelationshipType
                                , `AccountStatus`
                                , `Total Legal Claim Contract`
                                , `Total Legal Claim Account`
                                , LegalEntity
                                , `Ένδειξη Ατομικής επιχείρησης`
                                , `Account Type (Master / Slave / Other)`
                                , DeceasedFlag
                                , `Agreement Status (Active/Inactive)`
                        from base
                        where `Agreement Status (Active/Inactive)` = 'Inactive'
                        or `Account Type (Master / Slave / Other)` = 'Slave'
                    """)

base_exclusion.createOrReplaceTempView("base_exclusion")

In [0]:
CDISlaveAccounts = spark.sql(f"""
                                select distinct
                                          PortfolioName
                                        , CIF
                                        , AccountNumber
                                        , ContractNumber
                                        , CaseCode
                                        , RelationshipType
                                        , `AccountStatus`
                                        , `Total Legal Claim Contract`
                                        , `Total Legal Claim Account`
                                        , LegalEntity
                                        , `Ένδειξη Ατομικής επιχείρησης`
                                        , `Account Type (Master / Slave / Other)`
                                        , DeceasedFlag
                                        , 'Slave Account' `Exclusion Reason`
                                from base_exclusion
                                where `Account Type (Master / Slave / Other)` = 'Slave'
                            """)

CDISlaveAccounts.createOrReplaceTempView("CDISlaveAccounts")

In [0]:
CDIClosedAgreements = spark.sql(f"""
                                select distinct
                                          PortfolioName
                                        , CIF
                                        , AccountNumber
                                        , ContractNumber
                                        , CaseCode
                                        , RelationshipType
                                        , `AccountStatus`
                                        , `Total Legal Claim Contract`
                                        , `Total Legal Claim Account`
                                        , LegalEntity
                                        , `Ένδειξη Ατομικής επιχείρησης`
                                        , `Account Type (Master / Slave / Other)`
                                        , DeceasedFlag
                                        , 'Inactive Agreement' `Exclusion Reason`
                                from base_exclusion
                                where `Agreement Status (Active/Inactive)` = 'Inactive'
                            """)

CDIClosedAgreements.createOrReplaceTempView("CDIClosedAgreements")

In [0]:
closed_loans_sql = f"""
                            select distinct
                                      dp2.PortfolioName as PortfolioName
                                    , op.cdi as CIF
                                    , substring(op.exposurekey, 4, 50) as AccountNumber
                                    , substring(dex2.Agreementkey, 4, 50) as ContractNumber
                                    , cc.CaseCode
                                    , concat('Loan-', ex2.PartyRelationshipType, ' Rltp') as RelationshipType
                                    , coalesce(dex2.ExposureStatus, 'Closed') as `AccountStatus`
                                    , cast(null as decimal(29,2)) as `Total Legal Claim Contract`
                                    , cast(null as decimal(29,2)) as `Total Legal Claim Account`
                                    , dp2.PartyType as LegalEntity
                                    , CASE 
                                        WHEN dp2.LegalForm = 'ΙΔΙΩΤΕΣ - ΑΤΟΜΙΚΕΣ ΕΠΙΧΕΙΡΗΣΕΙΣ' AND dp2.PartyType = 'Organization' THEN 'Yes'
                                        ELSE 'No'
                                      END as `Ένδειξη Ατομικής επιχείρησης`
                                    , case 
                                        when s2.SLAVE_ACC is not null then 'Slave'
                                        when m2.MASTER_ACC is not null then 'Master'
                                        else 'Other'
                                      end as `Account Type (Master / Slave / Other)`
                                    , dp2.DeceasedFlag
                                    , 'Closed Loan' as `Exclusion Reason`
                            from offboarding_perimeter op
                            left join `{env}-cdh`.derivative.dparty dp2
                                on dp2.CDI = op.cdi
                                and dp2.BusinessDate = '{run_date}'
                            left join (
                                select Exposurekey, Agreementkey, ExposureStatus,
                                       row_number() over (partition by Exposurekey order by BusinessDate desc) rn
                                from `{env}-cdh`.derivative.dexposure
                                where Exposurekey in (select exposurekey from offboarding_perimeter)
                            ) dex2
                                on dex2.Exposurekey = op.exposurekey
                                and dex2.rn = 1
                            left join (
                                select Exposurekey, CDI, PartyRelationshipType,
                                       row_number() over (partition by Exposurekey, CDI order by BusinessDate desc) rn
                                from `{env}-cdh`.primary.exposure
                                where Exposurekey in (select exposurekey from offboarding_perimeter)
                            ) ex2
                                on ex2.Exposurekey = op.exposurekey
                                and ex2.CDI = op.cdi
                                and ex2.rn = 1
                            left join CaseCode cc
                                on cc.CDI = op.cdi
                                and (cc.ExposureKey = op.exposurekey or cc.ExposureKey = substring(op.exposurekey, 4, 50))
                            left join master m2
                                on op.exposurekey = m2.MASTER_ACC
                            left join slave s2
                                on op.exposurekey = s2.SLAVE_ACC
                            where not exists (
                                select 1 from base b
                                where b.CIF = op.cdi
                                and b.AccountNumber = substring(op.exposurekey, 4, 50)
                            )
                        """

# ClosedLoans is built from offboarding_perimeter, which only exists in perimeter mode.
if execution_mode == "perimeter":
    ClosedLoans = spark.sql(closed_loans_sql)
    ClosedLoans.createOrReplaceTempView("ClosedLoans")
    print(f"Closed Loans (not in exposure/dexposure): {ClosedLoans.count()} records")
else:
    ClosedLoans = None
    print("Portfolio mode - Closed Loans exclusion skipped (no offboarding perimeter)")

In [0]:
Exclusion_file = CDISlaveAccounts.union(CDIClosedAgreements)
if ClosedLoans is not None:
    Exclusion_file = Exclusion_file.union(ClosedLoans)

## Export to Volume

In [0]:
# notebook: {env}-cdh-delivery
# volume: {env}-cdh-delivery.internalreports.goodbyeletter
# export path: \\dwhsideapps\Share\in\{env}\InternalReports\goodbyeletter

In [0]:
from pyspark.sql.functions import col

# Define the output directory path
run_date_str = datetime.now().strftime("%Y%m%d")
output_dir = f"{report_output}{run_date_str}/" #f"/Volumes/d-cdh/reference/test/"

# Ensure the directory exists
os.makedirs(output_dir, exist_ok=True)

def export_to_excel(df, sheet_name):
    
    # Convert columns with dtype 'void' to string
    for col_name, dtype in df.dtypes:
        if dtype == 'void':
            df = df.withColumn(col_name, col(col_name).cast('string'))

    # Define the output file path
    output_filename = f"{output_file_prefix}_{sheet_name}_{run_date.strftime('%Y%m%d')}.xlsx"
    output_path = os.path.join(output_dir, output_filename)

    # Create an Excel writer object
    output_buffer = BytesIO()
    with pd.ExcelWriter(output_buffer, engine='openpyxl') as writer:
        # Save the subset to a temporary CSV file
        temp_file = os.path.join(output_dir, f"Goodbye_Letter.csv")
        df.coalesce(1).write.csv(temp_file, header=True, mode="overwrite")
        
        # Read the CSV back into Pandas
        csv_files = [f for f in os.listdir(temp_file) if f.endswith(".csv")]
        if csv_files:
            pandas_df = pd.read_csv(os.path.join(temp_file, csv_files[0]), dtype=str)
            
            # Write to an Excel sheet
            pandas_df.to_excel(writer, sheet_name=sheet_name, index=False)
        for file in os.listdir(temp_file):
            os.remove(os.path.join(temp_file, file))

    with open(output_path, "wb") as f:
        f.write(output_buffer.getvalue())

In [0]:
export_to_excel(Horizontal_file, 'Horizontal')
export_to_excel(Vertical_file, 'Vertical')
export_to_excel(Exclusion_file, 'Exclusion')